# Document Categorization — EDA and Training

This notebook is the reproducible evidence notebook required by the assignment. It performs EDA on the generated English/Spanish 20 Newsgroups splits, trains the baseline, fine-tunes multilingual DistilBERT for at least 5 epochs, and evaluates the final pipeline.

Run `python scripts/prepare_data.py` from the repository root before executing this notebook. No metric values are pre-filled.


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_processed_splits, dataset_summary

splits = load_processed_splits(ROOT / "data/processed_data")
train, validation, test = splits["train"], splits["validation"], splits["test"]
summary = dataset_summary(splits.values())
summary


## 1. Audit minimums

The prepared corpus must contain at least 10,000 documents, at least 5 categories, and at least 2 languages. These assertions deliberately fail if dataset preparation does not satisfy the subject.


In [ ]:
assert summary["documents"] >= 10_000, summary
assert summary["categories"] >= 5, summary
assert len(summary["languages"]) >= 2, summary
assert {"en", "es"}.issubset(summary["languages"]), summary
summary


## 2. Split sizes and class balance


In [ ]:
split_sizes = pd.Series({name: len(frame) for name, frame in splits.items()}, name="documents")
display(split_sizes.to_frame())
split_sizes.plot(kind="bar", title="Documents per split", ylabel="documents")
plt.tight_layout(); plt.show()


In [ ]:
category_counts = pd.concat(
    [frame["label"].value_counts().rename(name) for name, frame in splits.items()], axis=1
).fillna(0).astype(int)
display(category_counts)
category_counts.plot(kind="bar", figsize=(12,5), title="Category distribution by split")
plt.tight_layout(); plt.show()


## 3. Language balance and category × language coverage


In [ ]:
language_counts = pd.concat(
    [frame["language"].value_counts().rename(name) for name, frame in splits.items()], axis=1
).fillna(0).astype(int)
display(language_counts)
language_counts.plot(kind="bar", title="Language distribution by split")
plt.tight_layout(); plt.show()


In [ ]:
all_data = pd.concat([frame.assign(split=name) for name, frame in splits.items()], ignore_index=True)
pivot = pd.crosstab(all_data["label"], all_data["language"])
display(pivot)
assert (pivot > 0).all().all(), "Every category must be represented in every supported language"


## 4. Text quality, lengths and duplicates


In [ ]:
quality = pd.DataFrame({
    "missing_text": [int(frame["text"].isna().sum()) for frame in splits.values()],
    "empty_text": [int(frame["text"].fillna("").str.strip().eq("").sum()) for frame in splits.values()],
    "exact_duplicates": [int(frame["text"].duplicated().sum()) for frame in splits.values()],
}, index=splits.keys())
display(quality)
assert quality[["missing_text", "empty_text"]].to_numpy().sum() == 0


In [ ]:
sample = all_data.copy()
sample["words"] = sample["text"].str.split().str.len()
sample["characters"] = sample["text"].str.len()
display(sample.groupby("language")[["words", "characters"]].describe(percentiles=[.5,.9,.95,.99]))
for language, group in sample.groupby("language"):
    group["words"].clip(upper=group["words"].quantile(.99)).hist(alpha=.5, bins=50, label=language)
plt.legend(); plt.title("Document word-count distribution (clipped at p99)"); plt.xlabel("words"); plt.show()


## 5. Representative samples

Inspecting actual samples is useful for spotting translation artifacts, signatures, quoted content and source-specific noise.


In [ ]:
display(all_data.groupby(["language", "label"], group_keys=False).head(1)[["language", "label", "text"]].head(16))


## 6. Baseline — TF-IDF + Logistic Regression

This is the classical baseline used for the required ≥5 percentage-point transfer-learning comparison.


In [ ]:
from models.baseline import train_baseline

baseline = train_baseline(
    train, validation, ROOT / "models/checkpoints/baseline.joblib"
)
print(f"Validation accuracy: {baseline.accuracy:.4f}")
print(f"Validation macro F1: {baseline.f1_macro:.4f}")


## 7. Transfer learning — multilingual DistilBERT

Required settings: 5 epochs, learning rate 2e-5–5e-5, validation-loss monitoring, and a checkpoint after each epoch. The default below uses 5 epochs and 3e-5.


In [ ]:
from models.text_classifier import ClassifierConfig
from utils.transfer_learning import train_transformer

config = ClassifierConfig(epochs=5, learning_rate=3e-5, batch_size=16, max_length=256)
config.validate()
history = train_transformer(
    train, validation, ROOT / "models/checkpoints", config
)


## 8. Training curves and overfitting check


In [ ]:
history_df = pd.read_csv(ROOT / "models/checkpoints/training_history.csv")
display(history_df)
ax = history_df[["loss", "val_loss"]].plot(marker="o", title="Training vs validation loss")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); plt.show()
acc_cols = [c for c in ["accuracy", "val_accuracy"] if c in history_df]
if acc_cols:
    history_df[acc_cols].plot(marker="o", title="Training vs validation accuracy")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.show()


## 9. Final evaluation

Evaluation measures test accuracy, macro F1, English/Spanish accuracy, baseline improvement, and end-to-end batch throughput including spaCy tagging.


In [ ]:
import subprocess
subprocess.run([sys.executable, "scripts/evaluate.py"], cwd=ROOT, check=True)
metrics = json.loads((ROOT / "reports/performance_metrics.json").read_text())
display(pd.Series(metrics))


## 10. Audit thresholds


In [ ]:
assert metrics["classification_accuracy"] >= 0.85, metrics
assert metrics["f1_score_macro"] >= 0.80, metrics
assert metrics["processing_speed_docs_per_sec"] >= 100, metrics
assert all(v >= 0.80 for v in metrics["per_language_accuracy"].values()), metrics
assert metrics["accuracy_improvement_over_baseline"] >= 0.05, metrics
print("All mandatory performance thresholds passed.")


## 11. Optional quantization

The repository implements post-training TFLite dynamic-range quantization in `utils/model_optimization.py`. Run it after the unquantized model has passed accuracy validation; benchmark the optimized artifact separately before using it for production inference.


In [ ]:
# Optional: this may depend on TensorFlow Lite support for the installed Transformer graph.
# from utils.model_optimization import export_quantized_tflite
# export_quantized_tflite(ROOT / "models/checkpoints")
